# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² (FAIR2) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print key metadata - using attributes since .metadata is an object, not a dict
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}\n")
print(f"License: {dataset.metadata.license}")
print(f"Cite as: {getattr(dataset.metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, their unique `@id`s, and the fields/columns within each.

`mlcroissant` uses the Croissant metadata schema, so all references to data, fields, and columns use their `@id` for consistency.

Below we list all `RecordSet` objects (`@id`) and, for each, their columns and fields.

In [ ]:
# List all available record sets by @id and show details
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    print("  Fields (columns):")
    for field in rs.fields:
        col_ids = getattr(field, 'columns', [])
        col_id = col_ids[0] if col_ids else None
        print(f"    - Field @id: {field.id}, name: {field.name}, column: {col_id}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We'll use the record set and field `@id`s from the overview above.

For this dataset, the main data appears in the record set that contains proteomics results (`'cr:RecordSet/proteins'`). Update with the actual `@id` when visible in the previous cell.

In [ ]:
# For this dataset, let's extract all tabular record sets
# List all RecordSet @id for convenience
record_set_ids = [rs.id for rs in record_sets]
print(f"Available record set @id's:\n{record_set_ids}\n")

# Example: Use the main protein table. (You may need to adjust this depending on available @id's.)
# We'll arbitrarily select the first one for demonstration. (
# Replace with 'cr:RecordSet/proteins' or similar for actual exploration.
main_record_set_id = record_set_ids[0]

dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet {record_set_id}")

# Show all columns available for the main table
print(f"\nColumns for {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping using field and record set `@id`s only.

We will pick a numeric field, e.g., `coverage_percent`, by its `@id` (e.g., `'cr:Field/coverage_percent'` or the actual field `@id` found in the column list).

In [ ]:
# Identify a numeric field. If unsure, print columns and pick an appropriate one, e.g., 'cr:Field/coverage_percent'.
df = dataframes[main_record_set_id]
# Print sample columns for manual selection
print("Sample columns:", df.columns.tolist())

# Example: suppose the field @id for coverage percent is 'cr:Field/coverage_percent'
# For demonstration, select first numeric column
import numpy as np
numeric_field_id = None
for c in df.columns:
    # Try to coerce to numeric to find a numeric field
    try:
        if np.issubdtype(df[c].dropna().astype(float).dtype, np.number):
            numeric_field_id = c
            break
    except:
        continue

if numeric_field_id is None:
    print("No numeric field found to analyze.")
else:
    print(f"Numeric field selected: {numeric_field_id}")

    # Ensure the field is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter: Find records where value > 10 (as an example)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize the field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field (pick first object-type column)
    group_field_id = None
    for c in df.columns:
        if c != numeric_field_id and df[c].dtype == object:
            group_field_id = c
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the grouping field (if available).

This section demonstrates how to use the pandas and matplotlib libraries to plot data from the Croissant-driven DataFrame. All field references use the correct `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² Croissant dataset for mass spectrometry analysis using `mlcroissant`, referencing all fields by their `@id`. We loaded metadata, identified all record sets and fields, extracted data, performed basic EDA and normalization by field `@id`, and visualized key aspects of the dataset. For deeper domain insights, further expert curation and domain-specific feature engineering are recommended.